# Member 05 + Member 06 — Augmentation and Final Evaluation\n## CNN Patient-Age Regression on NIH ChestX-ray14\n\n**Kaggle dataset:** [Dataset NIH ChestX-ray14](https://www.kaggle.com/datasets/nguynhongmaivy/dataset-nih-chestx-ray14)\n\nNotebook này hoàn thiện hai phần cuối của coursework: thiết kế augmentation hợp lý cho ảnh X-quang (Member 05) và đánh giá cuối cùng trên test set, phân tích lỗi, bảng kết quả và kết luận (Member 06). Target thống nhất là `Patient Age`; tumor size không có ground truth đầy đủ trong NIH ChestX-ray14.

## Vai trò và câu hỏi thực nghiệm\n### Member 05 — Data augmentation\n- Chỉ augmentation tập train.\n- Dùng biến đổi nhẹ: resize, horizontal flip xác suất 0.5 và rotation ±7°.\n- Validation/test chỉ resize + normalize, hoàn toàn deterministic.\n- Kiểm tra trực quan ảnh trước/sau để tránh biến dạng giải phẫu.\n\n### Member 06 — Evaluation\n- Đánh giá trên cùng test split chưa từng dùng để chọn mô hình.\n- Báo cáo MAE (chính), RMSE, MSE và (R^2).\n- Vẽ predicted-vs-actual, residual distribution và sai số theo nhóm tuổi.\n- So sánh E1–E4 và viết kết luận dựa trên số đo, không dựa trên ảnh minh họa.

In [ ]:
# Cài đặt một lần nếu cần
# %pip install kagglehub torch torchvision pandas numpy scikit-learn matplotlib seaborn pillow tqdm

from pathlib import Path
import os, io, zipfile, random, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

SEED=42; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
IMG_SIZE=224; BATCH_SIZE=32
print('Device:', DEVICE)

## 1. Tải và tự động phát hiện dữ liệu Kaggle\nKaggleHub cần tài khoản Kaggle đã cấu hình token trong môi trường VS Code. Nếu nhóm đã tải thủ công, chỉ cần đặt `DATA_DIR` tới thư mục chứa CSV và ảnh. Không commit raw dataset vào repository.

In [ ]:
DATA_DIR=Path('data/nih-chestx-ray14')
if not DATA_DIR.exists():
    try:
        import kagglehub
        downloaded=Path(kagglehub.dataset_download('nguynhongmaivy/dataset-nih-chestx-ray14'))
        DATA_DIR=downloaded
        print('Downloaded to:', DATA_DIR)
    except Exception as e:
        print('KaggleHub download skipped:', e)

csv_candidates=list(DATA_DIR.rglob('*.csv'))
image_candidates=list(DATA_DIR.rglob('*.png'))
print('CSV files:', [str(p) for p in csv_candidates[:5]])
print('PNG count:', len(image_candidates))
assert csv_candidates, 'Không tìm thấy CSV. Hãy tải dataset Kaggle và đặt DATA_DIR đúng.'
CSV_PATH=next((p for p in csv_candidates if 'data_entry' in p.name.lower()),csv_candidates[0])
IMAGE_MAP={p.name:p for p in image_candidates}

In [ ]:
df=pd.read_csv(CSV_PATH)
age_col='Patient Age' if 'Patient Age' in df.columns else 'Age'
df['age']=pd.to_numeric(df[age_col].astype(str).str.extract(r'(\d+)')[0],errors='coerce')
df=df.dropna(subset=['age','Patient ID','Image Index'])
df=df[(df.age>=1)&(df.age<=100)].copy()
df['path']=df['Image Index'].map(IMAGE_MAP)
df=df[df.path.notna()].reset_index(drop=True)
print('Usable rows:',len(df),'| patients:',df['Patient ID'].nunique())
display(df[['Image Index','Patient ID','age','path']].head())

# Reuse fixed split CSVs if the repository has them; otherwise create a patient-wise split.
split_dir=Path('../../Member_02_Data/data/processed')
if all((split_dir/f'{k}.csv').exists() for k in ['train','val','test']):
    splits={k:pd.read_csv(split_dir/f'{k}.csv') for k in ['train','val','test']}
    for k in splits: splits[k]['age']=pd.to_numeric(splits[k]['Patient Age'],errors='coerce'); splits[k]['path']=splits[k]['Image Index'].map(IMAGE_MAP); splits[k]=splits[k][splits[k].path.notna()].reset_index(drop=True)
else:
    from sklearn.model_selection import GroupShuffleSplit
    g=GroupShuffleSplit(n_splits=1,test_size=.30,random_state=SEED); ti,vi=next(g.split(df,groups=df['Patient ID'])); tmp=df.iloc[vi]
    g2=GroupShuffleSplit(n_splits=1,test_size=.50,random_state=SEED); va,te=next(g2.split(tmp,groups=tmp['Patient ID']))
    splits={'train':df.iloc[ti].copy(),'val':tmp.iloc[va].copy(),'test':tmp.iloc[te].copy()}
ids={k:set(v['Patient ID']) for k,v in splits.items()}; assert not(ids['train']&ids['val'] or ids['train']&ids['test'] or ids['val']&ids['test'])
print({k:(len(v),v['Patient ID'].nunique()) for k,v in splits.items()})

## 2. Member 05 — Augmentation pipeline và kiểm tra trực quan\nHorizontal flip và rotation nhỏ mô phỏng sai khác khi thu nhận ảnh. Không dùng color jitter hoặc biến dạng mạnh vì ảnh X-quang grayscale và hình thái giải phẫu phải được bảo toàn.

In [ ]:
base_tf=transforms.Compose([transforms.Resize((IMG_SIZE,IMG_SIZE)),transforms.ToTensor(),transforms.Normalize([.5],[.5])])
aug_tf=transforms.Compose([transforms.Resize((IMG_SIZE,IMG_SIZE)),transforms.RandomHorizontalFlip(.5),transforms.RandomRotation(7),transforms.ToTensor(),transforms.Normalize([.5],[.5])])

sample=splits['train'].iloc[0]; original=Image.open(sample.path).convert('L')
fig,ax=plt.subplots(1,4,figsize=(14,4)); ax[0].imshow(original,cmap='gray'); ax[0].set_title('Original')
for i in range(1,4):
    t=aug_tf(original).squeeze().numpy(); t=(t*.5+.5).clip(0,1); ax[i].imshow(t,cmap='gray'); ax[i].set_title(f'Augmented {i}')
for a in ax:a.axis('off')
plt.suptitle('Training-only augmentation examples'); plt.show()

In [ ]:
class AgeDataset(Dataset):
    def __init__(self,frame,tf): self.frame=frame.reset_index(drop=True); self.tf=tf
    def __len__(self): return len(self.frame)
    def __getitem__(self,i):
        r=self.frame.iloc[i]; im=Image.open(r.path).convert('L'); return self.tf(im),torch.tensor(float(r.age)/100,dtype=torch.float32)
def loaders(use_aug):
    tr=DataLoader(AgeDataset(splits['train'],aug_tf if use_aug else base_tf),batch_size=BATCH_SIZE,shuffle=True,num_workers=0)
    va=DataLoader(AgeDataset(splits['val'],base_tf),batch_size=BATCH_SIZE,shuffle=False,num_workers=0)
    te=DataLoader(AgeDataset(splits['test'],base_tf),batch_size=BATCH_SIZE,shuffle=False,num_workers=0)
    return tr,va,te

## 3. CNN model và evaluation utilities
The architecture remains fixed across all experiments so that augmentation and loss are the only factors changed.

In [ ]:
class Block(nn.Module):
    def __init__(self,cin,cout): super().__init__(); self.net=nn.Sequential(nn.Conv2d(cin,cout,3,padding=1),nn.BatchNorm2d(cout),nn.ReLU(),nn.MaxPool2d(2))
    def forward(self,x): return self.net(x)
class CNNGAP(nn.Module):
    def __init__(self): super().__init__(); self.f=nn.Sequential(Block(1,32),Block(32,64),Block(64,128),Block(128,256)); self.gap=nn.AdaptiveAvgPool2d(1); self.out=nn.Linear(256,1)
    def forward(self,x): return self.out(self.gap(self.f(x)).flatten(1))

def pass_epoch(model,loader,loss_fn,opt=None):
    train=opt is not None; model.train(train); total=0; y=[]; p=[]
    for xb,yb in loader:
        xb,yb=xb.to(DEVICE),yb.to(DEVICE).flatten()
        with torch.set_grad_enabled(train):
            pred=model(xb).flatten(); loss=loss_fn(pred,yb)
            if train: opt.zero_grad(); loss.backward(); opt.step()
        total += loss.item()*len(xb); y.extend((yb.detach().cpu().numpy()*100)); p.extend((pred.detach().cpu().numpy()*100))
    return total/len(loader.dataset),mean_absolute_error(y,p),mean_squared_error(y,p)**.5,r2_score(y,p),np.array(y),np.array(p)

def train_eval(loss_name,use_aug,epochs=10):
    tr,va,te=loaders(use_aug); torch.manual_seed(SEED); model=CNNGAP().to(DEVICE); loss_fn=nn.MSELoss() if loss_name=='MSE' else nn.L1Loss(); opt=torch.optim.AdamW(model.parameters(),lr=1e-3,weight_decay=1e-4); best=np.inf; state=None; hist=[]
    for ep in range(epochs):
        a=pass_epoch(model,tr,loss_fn,opt); b=pass_epoch(model,va,loss_fn); hist.append([ep+1,a[0],a[1],b[0],b[1],b[2]])
        if b[1]<best: best=b[1]; state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
    model.load_state_dict(state); test=pass_epoch(model,te,loss_fn)
    return model,pd.DataFrame(hist,columns=['epoch','train_loss','train_mae','val_loss','val_mae','val_rmse']),test

## 4. Member 06 — E1–E4 final evaluation\nRun the four experiments with identical split, seed, model, optimizer, learning rate, batch size and epochs. Only `loss_name` or `use_aug` changes.

In [ ]:
RUN=False  # set True after confirming the data loader
all_results=[]; histories={}; predictions={}
if RUN:
    for loss_name,use_aug in [('MSE',False),('MAE',False),('MSE',True),('MAE',True)]:
        key=f'{loss_name}+{"Aug" if use_aug else "NoAug"}'; model,hist,test=train_eval(loss_name,use_aug,epochs=10); histories[key]=hist; predictions[key]=test
        all_results.append({'Experiment':key,'Test MAE (years)':test[1],'Test RMSE (years)':test[2],'Test MSE (years^2)':test[2]**2,'Test R2':test[3]})
    results_df=pd.DataFrame(all_results); display(results_df.sort_values('Test MAE (years)'))
else: print('RUN=False. Set RUN=True to train and evaluate E1–E4.')

In [ ]:
if RUN:
    best_key=results_df.sort_values('Test MAE (years)').iloc[0]['Experiment']; y,p=predictions[best_key][4],predictions[best_key][5]
    fig,ax=plt.subplots(1,3,figsize=(17,4))
    for key,h in histories.items(): ax[0].plot(h.epoch,h.val_mae,label=key)
    ax[0].set(xlabel='Epoch',ylabel='Validation MAE (years)',title='Validation MAE'); ax[0].legend()
    ax[1].scatter(y,p,s=5,alpha=.2); ax[1].plot([0,100],[0,100],'r--'); ax[1].set(xlabel='Actual age',ylabel='Predicted age',title=f'Best: {best_key}')
    sns.histplot(y-p,bins=40,kde=True,ax=ax[2]); ax[2].set(xlabel='Residual (actual − predicted)',title='Residual distribution'); plt.tight_layout(); plt.show()
    eval_df=splits['test'].copy(); eval_df['pred_age']=p; eval_df['abs_error']=abs(eval_df.age-eval_df.pred_age); eval_df['age_group']=pd.cut(eval_df.age,[0,18,40,60,80,100],right=False)
    display(eval_df.groupby('age_group',observed=True).abs_error.agg(['count','mean','median']))
    display(eval_df.nlargest(10,'abs_error')[['Image Index','age','pred_age','abs_error','Patient ID']])

## 5. Đánh giá và kết luận để đưa vào báo cáo\n### Quy tắc chọn mô hình\nChọn checkpoint theo validation MAE. Sau khi chốt checkpoint, báo cáo test MAE là sai số tuyệt đối trung bình theo **năm**; RMSE cũng theo năm; MSE theo năm². Không điều chỉnh hyperparameter sau khi xem test.\n\n### Mẫu diễn giải kết quả\n- Nếu `MSE+Aug` có MAE thấp nhất: augmentation giúp mô hình tổng quát hóa tốt hơn khi tối ưu MSE.\n- Nếu `MAE+NoAug` tốt nhất: augmentation hoặc phạt outlier bằng MSE không phù hợp với split này; cần báo cáo đúng kết quả thay vì giả định augmentation luôn tốt.\n- Nếu cả bốn mô hình gần baseline tuổi trung bình: tín hiệu hình ảnh yếu hoặc pipeline chưa đủ huấn luyện; không được tuyên bố mô hình hiệu quả.\n- Nếu residual lệch theo nhóm tuổi: dữ liệu mất cân bằng; cần cân nhắc sampling/weighted loss trong công việc tương lai.\n\n### Hạn chế và đạo đức\nNIH ChestX-ray14 là dữ liệu đơn trung tâm; nhãn bệnh yếu và phân phối có thể khác bệnh viện khác. Mô hình chỉ phục vụ coursework, không dùng để chẩn đoán hay quyết định điều trị. Tumor-size regression cần ảnh có annotation kích thước được chuyên gia xác nhận.

## 6. Checklist bàn giao Member 05–06\n- [ ] Augmentation chỉ bật cho train.\n- [ ] Ảnh augmentation được trực quan hóa.\n- [ ] Patient IDs giữa các split không giao nhau.\n- [ ] E1–E4 dùng cùng seed và hyperparameters.\n- [ ] Chỉ chọn mô hình bằng validation MAE.\n- [ ] Test MAE, RMSE, MSE và (R^2) được ghi rõ đơn vị.\n- [ ] Có predicted-vs-actual, residual và lỗi theo nhóm tuổi.\n- [ ] Có baseline tuổi trung bình.\n- [ ] Có limitation/ethics và không tuyên bố clinical deployment.\n\n**Nguồn:** [Kaggle dataset](https://www.kaggle.com/datasets/nguynhongmaivy/dataset-nih-chestx-ray14), [NIH download page](https://nihcc.app.box.com/v/ChestXray-NIHCC), [NIH dataset documentation](https://docs.cloud.google.com/healthcare-api/docs/resources/public-datasets/nih-chest).